# 01 Raw Data Audit

## AFL Matchday Demand Forecasting

**Purpose:** Validate the identity, integrity, schema, coverage, and known quality issues of the immutable raw data assets before any staging transformations are applied.

**Scope:**

- Kaggle AFL Stats Version 6
- Squiggle 2026 games and teams snapshots
- Figshare school-holiday data for Australian capital cities

**Data policy:** Files under `data/raw/` are immutable source records. This notebook may read and diagnose them, but it must not modify, overwrite, deduplicate, or correct any raw file.

**Kernel:** `Python (AFL_env)`


## 1. Environment and Project Paths

This section imports the audit dependencies, locates the project root, and defines every raw input as an explicit path. The project-root search allows the notebook to run whether VS Code starts the kernel from the repository root or from the `notebooks/` directory.


In [11]:
from __future__ import annotations

# Import standard-library modules for hashing, JSON parsing, path handling,
# archive validation, and runtime diagnostics.
import hashlib
import json
import sys
import csv
from pathlib import Path
from zipfile import ZipFile

# Import third-party analysis and notebook-display utilities.
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    '''Return the nearest parent directory containing the expected raw-data folder.'''
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "raw").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook from the AFL_Project workspace."
    )


# Resolve the repository root from the kernel's current working directory.
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Define every immutable source asset explicitly to prevent ambiguous file discovery.
RAW_PATHS = {
    "squiggle_games_json": RAW_DIR / "api_snapshots" / "squiggle_games_2026_20260817.json",
    "squiggle_teams_json": RAW_DIR / "api_snapshots" / "squiggle_teams_20260817.json",
    "kaggle_games_csv": RAW_DIR / "extracted" / "aflstats_v6" / "games.csv",
    "kaggle_players_csv": RAW_DIR / "extracted" / "aflstats_v6" / "players.csv",
    "kaggle_stats_csv": RAW_DIR / "extracted" / "aflstats_v6" / "stats.csv",
    "school_holidays_rda": RAW_DIR / "extracted" / "school_holidays_figshare_v1" / "school.holidays.rda",
    "school_holidays_txt": RAW_DIR / "extracted" / "school_holidays_figshare_v1" / "school.holidays.txt",
    "aflstats_zip": RAW_DIR / "packages" / "aflstats_v6.zip",
    "school_holidays_zip": RAW_DIR / "packages" / "school_holidays_figshare_v1.zip",
}

# Display the active interpreter and resolved paths before any data is read.
print(f"Python executable: {sys.executable}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Raw data folder:   {RAW_DIR}")


Python executable: d:\Projects\AFL_Project\AFL_venv\Scripts\python.exe
Project root:      D:\Projects\AFL_Project
Raw data folder:   D:\Projects\AFL_Project\data\raw


## 2. Raw File Inventory and Checksums

The inventory confirms that all expected assets exist and are non-empty. SHA-256 values are calculated from the local bytes so the exact source snapshots can later be recorded in an ingestion manifest.


In [2]:
def calculate_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    '''Calculate a SHA-256 digest without loading the entire file into memory.'''
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Build one inventory record for each expected raw asset.
inventory_records = []

for asset_name, path in RAW_PATHS.items():
    exists = path.is_file()
    size_bytes = path.stat().st_size if exists else None
    sha256 = calculate_sha256(path) if exists else None

    inventory_records.append(
        {
            "asset_name": asset_name,
            "relative_path": str(path.relative_to(PROJECT_ROOT)),
            "exists": exists,
            "size_bytes": size_bytes,
            "sha256": sha256,
        }
    )

# Present file identity, byte size, and checksum as a reviewable audit table.
inventory_df = pd.DataFrame(inventory_records)
display(inventory_df)

# Fail immediately when an expected raw asset is missing or empty.
assert inventory_df["exists"].all(), "One or more expected raw files are missing."
assert inventory_df["size_bytes"].gt(0).all(), "One or more raw files are empty."

# Kaggle Version 6 is frozen at this known byte size.
observed_games_size = int(
    inventory_df.loc[
        inventory_df["asset_name"].eq("kaggle_games_csv"),
        "size_bytes",
    ].iloc[0]
)
assert observed_games_size == 403_987, "games.csv does not match the frozen Version 6 byte size."


,asset_name,relative_path,exists,size_bytes,sha256
0,squiggle_games_json,data\raw\api_snapshots\squiggle_games_2026_202...,True,100507,c926b8ad99aab09cb949877ba69ae33854d6200dc59dbb...
1,squiggle_teams_json,data\raw\api_snapshots\squiggle_teams_20260817...,True,2541,40b08361c5c879195d35b8427cb8fa0d850644941125a7...
2,kaggle_games_csv,data\raw\extracted\aflstats_v6\games.csv,True,403987,ae1382f6a8440747b9c43855c41fa063b7ebde323570bc...
3,kaggle_players_csv,data\raw\extracted\aflstats_v6\players.csv,True,98444,f3335996d840c574fa8e772e649e20a66c25f08a1be8da...
4,kaggle_stats_csv,data\raw\extracted\aflstats_v6\stats.csv,True,14792829,31a261233cc4b7303ee5825c32ccb09bc509b77c0c6996...
5,school_holidays_rda,data\raw\extracted\school_holidays_figshare_v1...,True,142699,186b296431059688d657b4c1197b342c303b212a264034...
6,school_holidays_txt,data\raw\extracted\school_holidays_figshare_v1...,True,2151214,04282d576fb03b19970e67af7eb736f473c8722c7ddde6...
7,aflstats_zip,data\raw\packages\aflstats_v6.zip,True,2978718,ad847d4d9c8240bf5aed324733aa59f44330b2b4295669...
8,school_holidays_zip,data\raw\packages\school_holidays_figshare_v1.zip,True,2294239,e059169fb46441a7abeab4be44bf8f0bd8252a10749a6b...


### 2.1 ZIP Package Integrity

Each retained source package is tested with CRC verification. Archive members are listed for traceability; no files are extracted or modified by this check.


In [3]:
# Validate each retained download package without extracting or rewriting it.
package_results = []

for package_key in ("aflstats_zip", "school_holidays_zip"):
    package_path = RAW_PATHS[package_key]

    with ZipFile(package_path) as archive:
        corrupt_member = archive.testzip()
        member_names = archive.namelist()

    package_results.append(
        {
            "package": package_path.name,
            "status": "PASS" if corrupt_member is None else "FAIL",
            "corrupt_member": corrupt_member,
            "member_count": len(member_names),
            "members": ", ".join(member_names),
        }
    )

# Summarize archive status and retained member names for lineage review.
package_audit_df = pd.DataFrame(package_results)
display(package_audit_df)

# A corrupt archive blocks all downstream loading.
assert package_audit_df["status"].eq("PASS").all(), "At least one ZIP package is corrupt."


,package,status,corrupt_member,member_count,members
0,aflstats_v6.zip,PASS,None,3,"games.csv, players.csv, stats.csv"
1,school_holidays_figshare_v1.zip,PASS,None,2,"school.holidays.rda, school.holidays.txt"


## 3. Squiggle API Snapshot Validation

This section verifies that each saved API response is valid JSON, contains the expected top-level collection, and provides the minimum fields required for fixture and team-reference processing. It also checks record-ID uniqueness within each snapshot.


In [4]:
def audit_json_snapshot(
    path: Path,
    collection_key: str,
    required_fields: set[str],
) -> tuple[list[dict], dict]:
    '''Load one JSON snapshot and return its records and audit metrics.'''
    # Decode the complete saved response and validate its top-level structure.
    payload = json.loads(path.read_text(encoding="utf-8-sig"))

    if not isinstance(payload, dict):
        raise TypeError(f"{path.name} must contain a top-level JSON object.")

    records = payload.get(collection_key)

    if not isinstance(records, list):
        raise TypeError(
            f"{path.name} must contain a list under the '{collection_key}' key."
        )

    # Count records that cannot satisfy the minimum downstream schema contract.
    records_missing_required_fields = sum(
        not required_fields.issubset(record)
        for record in records
    )

    # Test provider identifiers for uniqueness within the individual snapshot.
    record_ids = [record.get("id") for record in records if record.get("id") is not None]
    duplicate_record_ids = len(record_ids) - len(set(record_ids))

    metrics = {
        "file": path.name,
        "collection": collection_key,
        "records": len(records),
        "records_missing_required_fields": records_missing_required_fields,
        "duplicate_record_ids": duplicate_record_ids,
    }

    return records, metrics


# Audit the fixture snapshot using match-level identity and scheduling fields.
squiggle_games, games_json_metrics = audit_json_snapshot(
    RAW_PATHS["squiggle_games_json"],
    collection_key="games",
    required_fields={"id", "year", "round", "date", "hteam", "ateam", "venue"},
)

# Audit the team reference snapshot using stable provider identifiers and names.
squiggle_teams, teams_json_metrics = audit_json_snapshot(
    RAW_PATHS["squiggle_teams_json"],
    collection_key="teams",
    required_fields={"id", "name", "abbrev"},
)

# Consolidate both API checks into one comparable result table.
squiggle_audit_df = pd.DataFrame([games_json_metrics, teams_json_metrics])
display(squiggle_audit_df)

# Missing fields or duplicate provider IDs would make the snapshot unsafe to load.
assert squiggle_audit_df["records"].gt(0).all(), "A Squiggle snapshot contains no records."
assert squiggle_audit_df["records_missing_required_fields"].eq(0).all(), (
    "A Squiggle snapshot is missing required fields."
)
assert squiggle_audit_df["duplicate_record_ids"].eq(0).all(), (
    "A Squiggle snapshot contains duplicate provider record IDs."
)


,file,collection,records,records_missing_required_fields,duplicate_record_ids
0,squiggle_games_2026_20260817.json,games,218,0,0
1,squiggle_teams_20260817.json,teams,18,0,0


## 4. Kaggle Match History Audit

### 4.1 Schema, Coverage, Dates, and Attendance

The Kaggle `games.csv` file is the historical match and attendance source for Version 1. This check confirms the frozen dimensions, required schema, season coverage, valid dates, and parseable non-negative attendance values.


In [5]:
# Load the historical match source without altering provider values.
games_raw = pd.read_csv(
    RAW_PATHS["kaggle_games_csv"],
    dtype={"GameId": "string"},
    keep_default_na=True,
)

# Define the minimum schema required by staging and feature engineering.
required_game_columns = {
    "GameId",
    "Year",
    "Round",
    "Date",
    "Venue",
    "StartTime",
    "Attendance",
    "HomeTeam",
    "AwayTeam",
}

# Parse analytical fields into temporary audit series; the raw DataFrame remains unchanged.
missing_game_columns = sorted(required_game_columns - set(games_raw.columns))
parsed_game_dates = pd.to_datetime(games_raw["Date"], errors="coerce")
parsed_attendance = pd.to_numeric(
    games_raw["Attendance"].astype("string").str.replace(",", "", regex=False),
    errors="coerce",
)

# Collect deterministic acceptance metrics for the frozen Version 6 snapshot.
game_schema_metrics = {
    "rows": len(games_raw),
    "columns": len(games_raw.columns),
    "minimum_year": int(games_raw["Year"].min()),
    "maximum_year": int(games_raw["Year"].max()),
    "missing_required_columns": len(missing_game_columns),
    "invalid_dates": int(parsed_game_dates.isna().sum()),
    "unparseable_attendance": int(parsed_attendance.isna().sum()),
    "negative_attendance": int(parsed_attendance.lt(0).sum()),
    "fully_duplicated_rows": int(games_raw.duplicated().sum()),
}

# Display the observed schema and coverage metrics before enforcing assertions.
game_schema_audit_df = pd.DataFrame(
    game_schema_metrics.items(),
    columns=["check", "observed_value"],
)
display(game_schema_audit_df)
print("Columns:", ", ".join(games_raw.columns))

# These assertions define the frozen Kaggle Version 6 acceptance criteria.
assert games_raw.shape == (2_879, 22), "Unexpected games.csv dimensions."
assert game_schema_metrics["minimum_year"] == 2012, "Unexpected minimum season."
assert game_schema_metrics["maximum_year"] == 2025, "Unexpected maximum season."
assert not missing_game_columns, f"Missing required columns: {missing_game_columns}"
assert game_schema_metrics["invalid_dates"] == 0, "One or more match dates are invalid."
assert game_schema_metrics["unparseable_attendance"] == 0, (
    "One or more attendance values cannot be parsed."
)
assert game_schema_metrics["negative_attendance"] == 0, (
    "One or more attendance values are negative."
)


,check,observed_value
0,rows,2879
1,columns,22
2,minimum_year,2012
3,maximum_year,2025
4,missing_required_columns,0
5,invalid_dates,0
6,unparseable_attendance,0
7,negative_attendance,0
8,fully_duplicated_rows,0


Columns: GameId, Year, Round, Date, MaxTemp, MinTemp, Rainfall, Venue, StartTime, Attendance, HomeTeam, HomeTeamScoreQT, HomeTeamScoreHT, HomeTeamScore3QT, HomeTeamScoreFT, HomeTeamScore, AwayTeam, AwayTeamScoreQT, AwayTeamScoreHT, AwayTeamScore3QT, AwayTeamScoreFT, AwayTeamScore


### 4.2 `GameId` Uniqueness

A provider ID should be tested rather than assumed to be unique. The rows below isolate every repeated `GameId` and retain enough match context to distinguish an exact duplicate from an identifier collision.


In [6]:
# Isolate every row participating in a provider-ID collision.
duplicate_game_id_rows = (
    games_raw.loc[
        games_raw["GameId"].duplicated(keep=False),
        [
            "GameId",
            "Year",
            "Round",
            "Date",
            "StartTime",
            "HomeTeam",
            "AwayTeam",
            "Venue",
            "Attendance",
        ],
    ]
    .sort_values(["GameId", "Date", "StartTime"])
    .reset_index(drop=True)
)

# Separate affected rows, distinct duplicated IDs, and excess occurrences.
duplicate_id_metrics = {
    "rows_with_duplicated_game_id": len(duplicate_game_id_rows),
    "distinct_duplicated_game_ids": duplicate_game_id_rows["GameId"].nunique(),
    "duplicate_occurrences_beyond_unique": (
        len(games_raw) - games_raw["GameId"].nunique()
    ),
}

# Show both the aggregate collision metrics and the underlying match identities.
display(pd.DataFrame(duplicate_id_metrics.items(), columns=["metric", "value"]))
display(duplicate_game_id_rows)

# Version 6 contains three known provider-ID collisions, not duplicate matches.
assert duplicate_id_metrics["duplicate_occurrences_beyond_unique"] == 3, (
    "The observed GameId collision count differs from the reviewed source snapshot."
)


,metric,value
0,rows_with_duplicated_game_id,6
1,distinct_duplicated_game_ids,3
2,duplicate_occurrences_beyond_unique,3


,GameId,Year,Round,Date,StartTime,HomeTeam,AwayTeam,Venue,Attendance
0,2024OR01,2024,Opening Round,2024-03-08,6:40 PM,Brisbane,Carlton,Gabba,"33,367"
1,2024OR01,2024,Opening Round,2024-03-14,7:30 PM,Carlton,Richmond,MCG,"83,881"
2,2024OR02,2024,Opening Round,2024-03-09,3:20 PM,Gold Coast,Richmond,Carrara,"22,086"
3,2024OR02,2024,Opening Round,2024-03-15,7:40 PM,Collingwood,Sydney,MCG,"78,933"
4,2024OR03,2024,Opening Round,2024-03-09,7:30 PM,Greater Western Sydney,Collingwood,Sydney Showground,"21,235"
5,2024OR03,2024,Opening Round,2024-03-16,1:45 PM,Essendon,Hawthorn,MCG,"73,805"


### Duplicate Game Identifier Assessment

The audit identified six records associated with three duplicated `GameId` values: `2024OR01`, `2024OR02`, and `2024OR03`. Each identifier occurs twice.  
  
However, the corresponding records represent different matches, as confirmed by their dates, participating teams, venues, and attendance figures.

These observations indicate the **identifier collisions rather than duplicated match records**. The affected records should not be removed using a simple `GameId`-based deduplication rule.

The collisions are associated with the 2024 transition from the Opening Round to Round 1. Some Round 1 matches appear to retain the `Opening Round` label and reuse identifiers previously assigned to genuine Opening Round fixtures.

The raw source data will remain unchanged. During the staging process:

1. the original `GameId` will be retained as `source_game_id`;
2. the affected round labels will be corrected using an explicitly documented transformation rule;
3. a canonical match identifier will be constructed from validated match attributes; and
4. uniqueness will be reassessed after the correction.

This approach preserves source lineage while preventing incorrect record deletion and downstream join ambiguity.

### 4.3 Known 2024 Round-Labelling Defect

The 2024 source records combine the four true Opening Round matches with all nine Round 1 matches under the label `Opening Round`. This also reuses `2024OR01`, `2024OR02`, and `2024OR03` for different matches.

**Decision:** Preserve the raw values exactly. A staging correction register will relabel the nine matches played from 14 to 17 March 2024 as `Round 1`. The canonical match key will be derived from normalized match identity fields rather than the non-unique source `GameId`.


In [7]:
# Create an in-memory audit view with parsed dates; do not write changes to raw data.
games_audit = games_raw.assign(DateParsed=parsed_game_dates)

# Review the complete 2024 Opening Round and Round 1 calendar window.
round_review = (
    games_audit.loc[
        games_audit["Year"].eq(2024)
        & games_audit["DateParsed"].between("2024-03-07", "2024-03-17"),
        [
            "DateParsed",
            "StartTime",
            "GameId",
            "Round",
            "HomeTeam",
            "AwayTeam",
        ],
    ]
    .sort_values(["DateParsed", "StartTime"])
    .reset_index(drop=True)
)

# Partition the correctly labelled opening fixtures from the mislabeled Round 1 fixtures.
true_opening_round = round_review[round_review["DateParsed"].le("2024-03-09")]
round_one_candidates = round_review[round_review["DateParsed"].ge("2024-03-14")]

# Quantify the anomaly before defining a downstream correction register.
round_issue_metrics = {
    "matches_labelled_opening_round_in_window": int(
        round_review["Round"].eq("Opening Round").sum()
    ),
    "true_opening_round_matches": len(true_opening_round),
    "round_one_matches_requiring_staging_correction": len(round_one_candidates),
}

# Present the aggregate finding together with every affected match.
display(pd.DataFrame(round_issue_metrics.items(), columns=["metric", "value"]))
display(round_review)

# Confirm the reviewed defect before the correction rule is implemented downstream.
assert len(true_opening_round) == 4, "Unexpected number of true Opening Round matches."
assert len(round_one_candidates) == 9, "Unexpected number of Round 1 correction candidates."
assert round_one_candidates["Round"].eq("Opening Round").all(), (
    "Not all reviewed Round 1 candidates carry the expected erroneous source label."
)


,metric,value
0,matches_labelled_opening_round_in_window,13
1,true_opening_round_matches,4
2,round_one_matches_requiring_staging_correction,9


,DateParsed,StartTime,GameId,Round,HomeTeam,AwayTeam
0,2024-03-07,7:30 PM,2024OOR,Opening Round,Sydney,Melbourne
1,2024-03-08,6:40 PM,2024OR01,Opening Round,Brisbane,Carlton
2,2024-03-09,3:20 PM,2024OR02,Opening Round,Gold Coast,Richmond
3,2024-03-09,7:30 PM,2024OR03,Opening Round,Greater Western Sydney,Collingwood
4,2024-03-14,7:30 PM,2024OR01,Opening Round,Carlton,Richmond
5,2024-03-15,7:40 PM,2024OR02,Opening Round,Collingwood,Sydney
6,2024-03-16,1:45 PM,2024OR03,Opening Round,Essendon,Hawthorn
7,2024-03-16,4:35 PM,2024OR04,Opening Round,Greater Western Sydney,North Melbourne
8,2024-03-16,7:10 PM,2024OR06,Opening Round,Gold Coast,Adelaide
9,2024-03-16,7:30 PM,2024OR05,Opening Round,Geelong,St Kilda


### Assessment of the 2024 Opening Round Label Anomaly

The source data contains 13 matches labelled as `Opening Round` between 7 March and 17 March 2024. Examination of the fixture sequence shows that these records represent two separate competition rounds:

- four genuine Opening Round matches played from 7 to 9 March 2024; and
- nine Round 1 matches played from 14 to 17 March 2024.

The nine later matches therefore require a controlled round-label correction during staging. This systematic labelling anomaly also explains the previously identified `GameId` collisions: the identifiers assigned to several Round 1 matches reuse values already assigned to genuine Opening Round fixtures.

The raw fields will remain unchanged for auditability. The staging layer will preserve the original values as `source_game_id` and `source_round`, while assigning `Round 1` as the standardised round label for the affected records. The correction will be restricted to the validated 2024 date window to avoid unintentionally altering records from other seasons.

After transformation, the corrected match count, round distribution, and canonical identifier uniqueness will be validated before the data is used in downstream analysis or modelling.

## 5. Figshare School-Holiday Data Audit

### 5.1 Structure, Coverage, and Key Uniqueness

`school.holidays.txt` is a comma-delimited text file with a source row-number column. The audit verifies its eight-city daily grid, date coverage, `(City, Date)` uniqueness, and holiday-code domains. The companion `.rda` file remains preserved as an immutable source asset but is not required for the Python audit.


#### Raw Text Structure Preview

The raw text structure is inspected before pandas parsing to determine whether the header and data records contain the same number of fields. This provides direct evidence for recovering the unnamed leading field as a source row identifier.

In [12]:
# Inspect the raw delimiter structure before pandas performs index inference.
header_fields = []
sample_rows = []
data_row_widths = set()
data_row_count = 0

with RAW_PATHS["school_holidays_txt"].open(
    mode="r",
    encoding="utf-8-sig",
    newline="",
) as source_file:
    reader = csv.reader(source_file)
    header_fields = next(reader)

    # Check the field width across the complete file while retaining only
    # five representative records for display.
    for row in reader:
        data_row_count += 1
        data_row_widths.add(len(row))

        if len(sample_rows) < 5:
            sample_rows.append(row)

# Confirm that the header contains four named fields while every data
# record contains one additional unnamed leading value.
assert len(header_fields) == 4, (
    f"Expected four header fields, observed {len(header_fields)}."
)
assert data_row_widths == {5}, (
    f"Inconsistent data-row widths detected: {sorted(data_row_widths)}"
)
assert data_row_count == 58_440, (
    f"Expected 58,440 data rows, observed {data_row_count:,}."
)

# Display a compact structural summary and representative raw records.
structure_summary = pd.DataFrame(
    {
        "record_type": ["header", "data_rows"],
        "field_count": [len(header_fields), 5],
        "record_count": [1, data_row_count],
    }
)

raw_preview = pd.DataFrame(
    sample_rows,
    columns=["unnamed_leading_field", *header_fields],
)

display(structure_summary)
display(raw_preview)

,record_type,field_count,record_count
0,header,4,1
1,data_rows,5,58440


,unnamed_leading_field,City,Date,schoolhols,school.hols
0,1,Adelaide,2004-01-01,1,4
1,2,Brisbane,2004-01-01,1,4
2,3,Canberra,2004-01-01,1,4
3,4,Darwin,2004-01-01,1,4
4,5,Hobart,2004-01-01,1,4


#### Parsing Decision

The raw-structure inspection shows that the file header contains four named fields, while all 58,440 data records contain five values. The preview indicates that the additional leading value is a source-level record number rather than an analytical variable.

Because this leading field has no corresponding header name, pandas interprets it automatically as the DataFrame index. The following parsing step will therefore recover the inferred index as an explicit `source_row_id` column using `rename_axis()` and `reset_index()`.

The recovered identifier will be retained for record-level traceability but excluded from analytical feature engineering. The original text file will remain unchanged. After recovery, the resulting five-column source schema will be validated before date parsing, coverage assessment, and natural-key testing are performed.

In [13]:
# Read the text export as strings and preserve the source-level "NA" token.
holidays_raw = pd.read_csv(
    RAW_PATHS["school_holidays_txt"],
    dtype="string",
    keep_default_na=False,
)

# Recover the unnamed leading record number inferred by pandas as the index.
holidays_audit = (
    holidays_raw
    .rename_axis("source_row_id")
    .reset_index()
)
holidays_audit.columns = holidays_audit.columns.str.strip()

# Validate the exact source schema before referencing individual fields.
expected_columns = [
    "source_row_id",
    "City",
    "Date",
    "schoolhols",
    "school.hols",
]

assert holidays_audit.columns.tolist() == expected_columns, (
    f"Unexpected school-holiday schema: {holidays_audit.columns.tolist()}"
)

source_column_count = holidays_audit.shape[1]

# Parse dates in the audit view without modifying the immutable raw file.
holidays_audit["DateParsed"] = pd.to_datetime(
    holidays_audit["Date"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Normalise the two source code fields for validation only.
schoolhols_codes = (
    holidays_audit["schoolhols"]
    .fillna("")
    .str.strip()
    .str.upper()
)
school_hols_codes = (
    holidays_audit["school.hols"]
    .fillna("")
    .str.strip()
    .str.upper()
)

# Treat blank strings and the retained "NA" token as missing values.
missing_tokens = {"", "NA"}
missing_schoolhols_mask = schoolhols_codes.isin(missing_tokens)
missing_school_hols_mask = school_hols_codes.isin(missing_tokens)

missing_schoolhols = int(missing_schoolhols_mask.sum())
missing_school_hols = int(missing_school_hols_mask.sum())
invalid_dates = int(holidays_audit["DateParsed"].isna().sum())

# Measure city-level coverage and test the natural (City, Date) key.
city_counts = (
    holidays_audit
    .groupby("City", dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("City")
    .reset_index(drop=True)
)

duplicate_city_date_rows = int(
    holidays_audit
    .duplicated(["City", "Date"], keep=False)
    .sum()
)

minimum_date = (
    holidays_audit["DateParsed"]
    .min()
    .date()
    .isoformat()
)
maximum_date = (
    holidays_audit["DateParsed"]
    .max()
    .date()
    .isoformat()
)

# Summarise the principal structural and data-quality observations.
holiday_metrics = {
    "rows": len(holidays_audit),
    "columns_including_source_row_id": source_column_count,
    "cities": holidays_audit["City"].nunique(),
    "minimum_date": minimum_date,
    "maximum_date": maximum_date,
    "invalid_dates": invalid_dates,
    "duplicate_city_date_rows": duplicate_city_date_rows,
    "missing_schoolhols": missing_schoolhols,
    "missing_school.hols": missing_school_hols,
    "schoolhols_values": sorted(schoolhols_codes.unique()),
    "school.hols_values": sorted(school_hols_codes.unique()),
}

display(
    pd.DataFrame(
        holiday_metrics.items(),
        columns=["check", "observed_value"],
    )
)
display(city_counts)

# Validate the frozen Figshare snapshot against its reviewed expectations.
expected_cities = {
    "Adelaide",
    "Brisbane",
    "Canberra",
    "Darwin",
    "Hobart",
    "Melbourne",
    "Perth",
    "Sydney",
}

assert len(holidays_audit) == 58_440, (
    "Unexpected school-holiday row count."
)
assert source_column_count == 5, (
    "Unexpected number of source columns."
)
assert set(holidays_audit["City"].unique()) == expected_cities, (
    "Unexpected capital-city coverage."
)
assert city_counts["rows"].eq(7_305).all(), (
    "At least one city has incomplete daily coverage."
)
assert minimum_date == "2004-01-01", (
    "Unexpected minimum date."
)
assert maximum_date == "2023-12-31", (
    "Unexpected maximum date."
)
assert invalid_dates == 0, (
    "One or more dates could not be parsed."
)
assert duplicate_city_date_rows == 0, (
    "Duplicate (City, Date) records were detected."
)
assert missing_schoolhols == 0, (
    "The schoolhols field contains missing values."
)
assert missing_school_hols == 1, (
    "The reviewed school.hols anomaly was not reproduced."
)
assert set(schoolhols_codes) == {"0", "1"}, (
    "Unexpected schoolhols code domain."
)
assert set(
    school_hols_codes.loc[~missing_school_hols_mask]
) == {"0", "1", "2", "3", "4"}, (
    "Unexpected school.hols code domain."
)

,check,observed_value
0,rows,58440
1,columns_including_source_row_id,5
2,cities,8
3,minimum_date,2004-01-01
4,maximum_date,2023-12-31
5,invalid_dates,0
6,duplicate_city_date_rows,0
7,missing_schoolhols,0
8,missing_school.hols,1
9,schoolhols_values,"[0, 1]"


,City,rows
0,Adelaide,7305
1,Brisbane,7305
2,Canberra,7305
3,Darwin,7305
4,Hobart,7305
5,Melbourne,7305
6,Perth,7305
7,Sydney,7305


#### Structural Audit Conclusion

The parsed dataset contains 58,440 records and five source fields, including the recovered `source_row_id`. All eight capital cities contain exactly 7,305 daily observations, producing a balanced panel covering 1 January 2004 to 31 December 2023.

No invalid dates or duplicated `(City, Date)` keys were detected. The `schoolhols` field is complete and restricted to the expected binary values of `0` and `1`.

The derived `school.hols` field contains the expected codes from `0` to `4`, together with one source-level `NA` value. This isolated anomaly will be examined in Section 5.2 and will not be silently imputed or used to modify the immutable raw source.

The dataset therefore passes the structural, temporal-coverage, and natural-key checks required for staging, subject to the documented treatment of the single missing derived holiday code.

### 5.2 Review of the Missing Derived Holiday Code

Section 5.1 identified one missing value in the derived `school.hols` field. The source represents this value using the literal token `NA`, rather than a blank field.

This section isolates the affected record and verifies that it corresponds to the previously reviewed Hobart observation dated 1 May 2022. The complete binary `schoolhols` indicator is also retained in the output to distinguish a missing derived category from a missing holiday-status value.

No imputation or source correction is performed during the raw-data audit.

In [14]:
# Reuse the missing-value mask created during the Section 5.1 audit.
missing_school_hols_rows = holidays_audit.loc[
    missing_school_hols_mask,
    [
        "source_row_id",
        "City",
        "Date",
        "schoolhols",
        "school.hols",
    ],
].reset_index(drop=True)

# Display the complete source record for anomaly review.
display(missing_school_hols_rows)

# Verify that the missing derived code matches the reviewed source anomaly.
expected_missing_row = missing_school_hols_rows.loc[
    missing_school_hols_rows["City"].eq("Hobart")
    & missing_school_hols_rows["Date"].eq("2022-05-01")
    & missing_school_hols_rows["schoolhols"].eq("1")
    & missing_school_hols_rows["school.hols"].str.upper().eq("NA")
]

assert len(missing_school_hols_rows) == 1, (
    "The number of missing school.hols values differs from the reviewed snapshot."
)
assert len(expected_missing_row) == 1, (
    "The reviewed Hobart 2022-05-01 anomaly was not reproduced."
)

,source_row_id,City,Date,schoolhols,school.hols
0,53565,Hobart,2022-05-01,1,NA


#### Treatment Decision

The anomaly affects only the derived `school.hols` category. The primary binary `schoolhols` field remains available and identifies the record as a school-holiday date.

The source-level `NA` token will remain unchanged in the immutable raw data. During staging, it will be converted to an explicit nullable value rather than being assigned an unsupported category. The complete binary `schoolhols` field can continue to support the planned school-holiday feature.

If the categorical `school.hols` field is required later, its treatment will be determined from verified source documentation or an explicitly documented derivation rule. Downstream join validation will also establish whether this city-date record corresponds to any AFL match in the modelling dataset.

## 6. Consolidated Audit Status


The detailed audit findings are consolidated below to determine whether the retained raw datasets may proceed to staging. A `PASS WITH WARNING` indicates that the data remains usable, provided that the identified issue receives a documented staging treatment.

| Audit area | Status | Finding |
|---|---|---|
| Raw file inventory and checksums | PASS | All nine expected raw assets are present, non-empty, and fingerprinted. |
| ZIP package integrity | PASS | Both retained source packages passed CRC verification. |
| Squiggle API snapshots | PASS | The 218 game records and 18 team records passed JSON structure and required-field checks. |
| Kaggle match-history schema and coverage | PASS | The expected 2,879 rows, 22 columns, and 2012–2025 coverage were reproduced. |
| Kaggle provider `GameId` | PASS WITH WARNING | Three provider identifiers are reused across six distinct match records and must not be used as the sole staging key. |
| 2024 round labels | PASS WITH WARNING | Nine Round 1 matches are incorrectly labelled as `Opening Round` and require a documented staging correction. |
| Figshare school-holiday panel | PASS WITH WARNING | Daily coverage is complete, but one derived `school.hols` value is represented by the source-level `NA` token. |

**Staging gate decision:** No blocking integrity failures were identified. The retained datasets may proceed to staging, subject to documented treatment of the provider-ID collisions, the 2024 round-label anomaly, and the single missing derived school-holiday code.

No raw source value will be modified as part of this decision.

## 7. Decisions and Next Steps

1. Preserve all files under `data/raw/` without modification.

2. Record source URLs, retrieval dates, dataset versions, file sizes, and SHA-256 checksums in an ingestion manifest.

3. Create a staging correction register covering:
   - the three reused Kaggle `GameId` values;
   - the nine 2024 Round 1 matches incorrectly labelled as `Opening Round`; and
   - the single source-level `NA` value in the derived `school.hols` field.

4. Retain the original Kaggle identifier as `source_game_id` and construct a canonical match key from validated match identity fields. The provider `GameId` must not be used as the sole primary key.

5. Verify the affected 2024 fixtures against a trusted independent source before implementing the round-label correction. A complete historical reconciliation is not required at this stage.

6. During staging, convert the source-level `NA` token in `school.hols` to an explicit nullable value. The complete binary `schoolhols` field will remain available for the planned school-holiday feature, while the derived category will not be imputed without a verified rule.

7. Once the notebook audit logic is stable, move only the deterministic and reusable validation checks into `src/validation/validate_raw_data.py`. The Notebook will remain the exploratory evidence and decision record.

No source correction is applied in this raw-data audit. The next project stage is the design of the staging schema, correction register, and canonical match key.